# Domain Bridge — Probability ↔ Bitstream ↔ Quantum

**SC-NeuroCore v3.14** — Seamless conversion between computational domains.

The `TensorStream` data structure provides automatic conversion
between three computational domains:

| Domain | Representation | Physics |
|--------|---------------|----------|
| `prob` | Scalar $p \in [0,1]$ | Classical probability |
| `bitstream` | Binary sequence $\{0,1\}^L$ | Stochastic computing |
| `quantum` | Amplitude $[\alpha, \beta]$ | Quantum state $|\psi\rangle$ |

The key invariant: probability is preserved across conversions.
The `QuantumStochasticLayer` adds a $\cos^2(\theta/2)$ non-linearity
via simulated qubit rotation — a quantum activation function.

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.core.tensor_stream import TensorStream
from sc_neurocore.quantum.hybrid import QuantumStochasticLayer

print("SC-NeuroCore domain bridge demo")

## 1. TensorStream: Probability → Bitstream → Probability

The roundtrip prob → bitstream → prob should recover the
original probability within $O(1/\sqrt{L})$.

In [ ]:
probs = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
ts = TensorStream.from_prob(probs)

print(f"Domain: {ts.domain}")
print(f"Data: {ts.data}")

# Convert to bitstream (L=4096)
np.random.seed(42)
bits = ts.to_bitstream(length=4096)
print(f"\nBitstream shape: {bits.shape}  (5 channels × 4096 bits)")

# Convert back to probability
ts_bits = TensorStream(data=bits, domain="bitstream")
recovered = ts_bits.to_prob()

print(f"\n{'Original':>10s}  {'Recovered':>10s}  {'Error':>10s}")
print("-" * 34)
for p, r in zip(probs, recovered):
    print(f"{p:10.3f}  {r:10.4f}  {abs(p - r):10.4f}")

## 2. Probability → Quantum State

The quantum encoding uses amplitude encoding:

$$|\psi\rangle = \sqrt{1-p}\,|0\rangle + \sqrt{p}\,|1\rangle$$

The Born rule gives $P(|1\rangle) = |\beta|^2 = p$, preserving
the stochastic computing probability exactly.

In [ ]:
quantum_states = ts.to_quantum()
print(f"Quantum state shape: {quantum_states.shape}  (5 qubits × [α, β])")
print()
print(f"{'p':>6s}  {'α (|0⟩)':>12s}  {'β (|1⟩)':>12s}  {'|β|²':>8s}  {'|α|²+|β|²':>10s}")
print("-" * 54)
for i, p in enumerate(probs):
    alpha = quantum_states[i, 0]
    beta = quantum_states[i, 1]
    born = abs(beta) ** 2
    norm = abs(alpha) ** 2 + abs(beta) ** 2
    print(f"{p:6.2f}  {alpha.real:12.6f}  {beta.real:12.6f}  {born:8.4f}  {norm:10.6f}")

## 3. Quantum → Probability (Born Rule)

Converting back from quantum to probability extracts $|\beta|^2$.

In [ ]:
ts_q = TensorStream(data=quantum_states, domain="quantum")
p_from_quantum = ts_q.to_prob()

print(f"Roundtrip prob → quantum → prob:")
np.testing.assert_allclose(probs, p_from_quantum, atol=1e-10)
print(f"  Max error: {np.max(np.abs(probs - p_from_quantum)):.2e}")
print(f"  Roundtrip is EXACT (no approximation in amplitude encoding).")

## 4. Quantum Stochastic Layer: $\cos^2(\theta/2)$ Non-linearity

The `QuantumStochasticLayer` simulates a qubit rotation gate:

$$p_{\text{in}} \xrightarrow{R_y(\theta)} p_{\text{out}} = \cos^2(\theta/2), \quad \theta = p_{\text{in}} \cdot \pi$$

This is a nonlinear activation function with quantum origin:
- $p=0 \rightarrow \cos^2(0) = 1.0$
- $p=0.5 \rightarrow \cos^2(\pi/4) \approx 0.854$
- $p=1 \rightarrow \cos^2(\pi/2) = 0.5$ (maximum uncertainty)

In [ ]:
qsl = QuantumStochasticLayer(n_qubits=5, length=8192)

# Create input bitstreams from our test probabilities
np.random.seed(42)
input_bits = ts.to_bitstream(length=8192)

output_bits = qsl.forward(input_bits)

p_in = np.mean(input_bits, axis=1)
p_out = np.mean(output_bits, axis=1)
p_expected = np.cos(p_in * np.pi / 2) ** 2

print(f"{'p_in':>8s}  {'p_out (SC)':>10s}  {'cos²(θ/2)':>10s}  {'error':>8s}")
print("-" * 40)
for pi, po, pe in zip(p_in, p_out, p_expected):
    print(f"{pi:8.3f}  {po:10.4f}  {pe:10.4f}  {abs(po - pe):8.4f}")

In [ ]:
# Sweep the full transfer function
p_sweep = np.linspace(0, 1, 50)
p_out_sweep = []

np.random.seed(42)
for p in p_sweep:
    bits_in = (np.random.random((1, 8192)) < p).astype(np.uint8)
    qsl_1 = QuantumStochasticLayer(n_qubits=1, length=8192)
    bits_out = qsl_1.forward(bits_in)
    p_out_sweep.append(np.mean(bits_out))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(p_sweep, p_out_sweep, "o", markersize=3, label="SC measurement")
ax.plot(p_sweep, np.cos(p_sweep * np.pi / 2) ** 2, "k-",
        linewidth=1.5, alpha=0.6, label=r"$\cos^2(p \cdot \pi / 2)$ (theory)")
ax.plot([0, 1], [0, 1], "gray", linestyle=":", alpha=0.3, label="identity")
ax.set_xlabel(r"Input probability $p_{in}$")
ax.set_ylabel(r"Output probability $p_{out}$")
ax.set_title("Quantum activation function: Ry rotation + measurement")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Conversion | Method | Error |
|-----------|--------|-------|
| prob → bitstream | Bernoulli sampling | $O(1/\sqrt{L})$ |
| bitstream → prob | Mean (popcount/L) | $O(1/\sqrt{L})$ |
| prob → quantum | Amplitude encoding | Exact |
| quantum → prob | Born rule $|\beta|^2$ | Exact |
| bitstream → quantum | Via prob | $O(1/\sqrt{L})$ |

The `TensorStream` abstraction enables seamless data flow
between classical stochastic computing, spiking neural networks,
and quantum circuits. The probability domain serves as the
universal intermediate representation.

The `QuantumStochasticLayer` adds a physically motivated
non-linearity ($\cos^2$) that operates directly on bitstreams —
no floating-point conversion needed in hardware.